# RQ3 modeling pipeline (age strand + disability strand)

Two modeling tasks per strand:
1. **Overall activity level** -- year x borough x group, predicting the
   inactive/fairly_active/active three-way share 
2. **Activity-specific participation** -- year x borough x group x activity,
   predicting participation for each individual activity, used to answer both
   "will this group participate in this activity" and "which activity has the highest predicted participation rate for this group".



## 0. Setup

In [43]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import json

RANDOM_STATE = 2026
ALR_EPSILON = 1e-6
ROLLING_ORIGINS = [4, 5, 6, 7]  # train on years < origin, validate on year == origin
AGE_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\q3")
DISABILITY_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed")

## 1. Shared functions


In [44]:
def shares_to_alr(shares):
    clipped = np.clip(np.asarray(shares, dtype=float), ALR_EPSILON, 1)
    clipped = clipped / clipped.sum(axis=1, keepdims=True)
    return np.log(clipped[:, :2] / clipped[:, [2]])


def alr_to_shares(values):
    values = np.clip(np.asarray(values, dtype=float), -30, 30)
    exponent = np.exp(values)
    denominator = 1 + exponent.sum(axis=1, keepdims=True)
    return np.column_stack([exponent / denominator, 1 / denominator])


In [45]:
def score_composition(actual, predicted, target_names):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    valid = np.isfinite(actual).all(axis=1) & np.isfinite(predicted).all(axis=1)
    actual, predicted = actual[valid], predicted[valid]
    row = {
        'observations': len(actual),
        'total_variation': np.mean(.5 * np.abs(actual - predicted).sum(axis=1)),
        'overall_mae': np.mean(np.abs(actual - predicted)),
        'overall_rmse': np.sqrt(np.mean((actual - predicted) ** 2)),
    }
    for index, target in enumerate(target_names):
        row[f'{target}_mae'] = mean_absolute_error(actual[:, index], predicted[:, index])
        row[f'{target}_rmse'] = np.sqrt(mean_squared_error(actual[:, index], predicted[:, index]))
        row[f'{target}_r2'] = r2_score(actual[:, index], predicted[:, index])
    return row


def score_single_rate(actual, predicted, target_name):
    actual = np.asarray(actual, dtype=float).ravel()
    predicted = np.asarray(predicted, dtype=float).ravel()
    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual, predicted = actual[valid], predicted[valid]
    return {
        'observations': len(actual),
        f'{target_name}_mae': mean_absolute_error(actual, predicted),
        f'{target_name}_rmse': np.sqrt(mean_squared_error(actual, predicted)),
        f'{target_name}_r2': r2_score(actual, predicted),
    }


In [46]:
def add_lag_features(frame, panel_keys, value_cols, lags=(1,), rolling_window=None, covid_years=(5, 6)):
    prepared = frame.sort_values(panel_keys + ['year']).reset_index(drop=True)
    grouped = prepared.groupby(panel_keys, sort=False)
    for column in value_cols:
        for lag in lags:
            prepared[f'{column}_lag{lag}'] = grouped[column].shift(lag)
        if rolling_window:
            prepared[f'{column}_roll{rolling_window}'] = grouped[column].transform(
                lambda s: s.shift(1).rolling(rolling_window, min_periods=1).mean()
            )
    prepared['time_trend'] = prepared['year'] / prepared['year'].max()
    prepared['is_covid_year'] = prepared['year'].isin(covid_years).astype(int)
    return prepared


def naive_predict(frame, lag_cols):
    return frame[lag_cols].to_numpy()

In [47]:
def add_interaction_terms(frame, group_col, time_col='time_trend'):
    dummies = pd.get_dummies(frame[group_col], prefix=f'{group_col}_x_time')
    interaction_cols = list(dummies.columns)
    frame[interaction_cols] = dummies.mul(frame[time_col], axis=0)
    return frame, interaction_cols

In [48]:
def parameter_candidates(model_name):
    if model_name == 'Ridge Regression':
        return [{'alpha': .1}, {'alpha': 1.0}, {'alpha': 10.0}, {'alpha': 100.0}]
    if model_name == 'Random Forest':
        return [
            {'n_estimators': 160, 'max_depth': 6, 'min_samples_leaf': 5, 'max_features': .5},
            {'n_estimators': 160, 'max_depth': 10, 'min_samples_leaf': 8, 'max_features': .5},
            {'n_estimators': 220, 'max_depth': 8, 'min_samples_leaf': 12, 'max_features': .8},
        ]
    return [
        {'n_estimators': 120, 'learning_rate': .05, 'max_depth': 2, 'min_samples_leaf': 15},
        {'n_estimators': 160, 'learning_rate': .03, 'max_depth': 2, 'min_samples_leaf': 20},
    ]


def build_model(model_name, parameters, numeric_features, categorical_features, multi_output):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if model_name == 'Ridge Regression':
        numeric_steps.append(('scale', StandardScaler()))
    preprocess = ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ])
    if model_name == 'Ridge Regression':
        estimator = Ridge(**parameters)
    elif model_name == 'Random Forest':
        estimator = RandomForestRegressor(**parameters, random_state=RANDOM_STATE, n_jobs=-1)
    else:
        base = GradientBoostingRegressor(**parameters, random_state=RANDOM_STATE, loss='huber')
        estimator = MultiOutputRegressor(base) if multi_output else base
    return Pipeline([('preprocess', preprocess), ('model', estimator)])


## 1b. Rolling-origin selection helpers

In [49]:
def get_eligible_split(prepared, target_cols, weight_col, lag_cols, before_year=None, at_year=None):
    if before_year is not None:
        rows = prepared[prepared['year'] < before_year]
        required = target_cols + [weight_col]
    elif at_year is not None:
        rows = prepared[prepared['year'] == at_year]
        required = target_cols + lag_cols
    else:
        raise ValueError('Provide exactly one of before_year or at_year.')
    return rows.dropna(subset=required).copy()


def rolling_origin_scores(
    prepared, target_cols, weight_col, lag_cols, numeric_for_model,
    categorical, model_name, params, origins, is_composition,
):
    origin_scores = []
    pooled_actual = []
    pooled_predicted = []

    for origin in origins:
        train = get_eligible_split(prepared, target_cols, weight_col, lag_cols, before_year=origin)
        val = get_eligible_split(prepared, target_cols, weight_col, lag_cols, at_year=origin)
        if train.empty or val.empty:
            continue

        model = build_model(model_name, params, numeric_for_model, categorical, multi_output=is_composition)
        train_weights = train[weight_col].to_numpy()

        if is_composition:
            model.fit(
                train[numeric_for_model + categorical],
                shares_to_alr(train[target_cols]),
                model__sample_weight=train_weights,
            )
            val_prediction = alr_to_shares(model.predict(val[numeric_for_model + categorical]))
            val_metrics = score_composition(val[target_cols], val_prediction, target_cols)
            primary_score = val_metrics['total_variation']
            pooled_actual.append(val[target_cols].to_numpy(dtype=float))
            pooled_predicted.append(val_prediction)
        else:
            target_col = target_cols[0]
            model.fit(
                train[numeric_for_model + categorical],
                train[target_col],
                model__sample_weight=train_weights,
            )
            val_prediction = np.clip(model.predict(val[numeric_for_model + categorical]), 0, 1)
            val_metrics = score_single_rate(val[target_col], val_prediction, target_col)
            primary_score = val_metrics[f'{target_col}_mae']
            pooled_actual.append(val[target_col].to_numpy(dtype=float))
            pooled_predicted.append(val_prediction)

        origin_scores.append(primary_score)

    if len(origin_scores) != len(origins):
        raise RuntimeError(
            f'{model_name}: expected {len(origins)} rolling origins, '
            f'but only {len(origin_scores)} were successfully evaluated.'
        )
    mean_score = float(np.mean(origin_scores)) if origin_scores else np.inf
    pooled_actual = np.concatenate(pooled_actual, axis=0) if pooled_actual else None
    pooled_predicted = np.concatenate(pooled_predicted, axis=0) if pooled_predicted else None
    return mean_score, origin_scores, pooled_actual, pooled_predicted


def select_best_params_rolling(
    prepared, target_cols, weight_col, lag_cols, numeric_by_model,
    categorical, model_names, origins, is_composition,
):
    selection = {}
    for model_name in model_names:
        numeric_for_model = numeric_by_model[model_name]
        best_params = None
        best_mean_score = np.inf
        best_origin_scores = None
        best_pooled_actual = None
        best_pooled_predicted = None

        for params in parameter_candidates(model_name):
            mean_score, origin_scores, pooled_actual, pooled_predicted = rolling_origin_scores(
                prepared, target_cols, weight_col, lag_cols, numeric_for_model,
                categorical, model_name, params, origins, is_composition,
            )
            if mean_score < best_mean_score:
                best_mean_score = mean_score
                best_params = params
                best_origin_scores = origin_scores
                best_pooled_actual = pooled_actual
                best_pooled_predicted = pooled_predicted

        if is_composition:
            pooled_metrics = score_composition(best_pooled_actual, best_pooled_predicted, target_cols)
        else:
            pooled_metrics = score_single_rate(best_pooled_actual, best_pooled_predicted, target_cols[0])

        selection[model_name] = {
            'best_params': best_params,
            'mean_validation_score': best_mean_score,
            'origin_scores': best_origin_scores,
            'pooled_validation_metrics': pooled_metrics,
        }
    return selection

## 2. Task type 1: three-way composition (inactive / fairly_active / active)


In [50]:
def run_composition_task(
    frame, panel_keys, target_cols, weight_col,
    origins=ROLLING_ORIGINS, test_year=8, feature_level='overall'
):
    if feature_level == 'overall':
        prepared = add_lag_features(frame, panel_keys, target_cols, lags=(1, 2))
        extra_cols = [f'{column}_lag2' for column in target_cols]
    else:
        prepared = add_lag_features(frame, panel_keys, target_cols, lags=(1,), rolling_window=2)
        extra_cols = [f'{column}_roll2' for column in target_cols]

    group_col = panel_keys[1]
    prepared, interaction_cols = add_interaction_terms(prepared, group_col)

    naive_lag_cols = [f'{column}_lag1' for column in target_cols]
    lag_cols = naive_lag_cols + extra_cols
    categorical = list(panel_keys)
    numeric = lag_cols + ['time_trend', 'is_covid_year']

    numeric_by_model = {
        'Ridge Regression': numeric + interaction_cols,
        'Random Forest': numeric,
        'Gradient Boosting': numeric,
    }

    trainval = get_eligible_split(prepared, target_cols, weight_col, naive_lag_cols, before_year=test_year)
    test = get_eligible_split(prepared, target_cols, weight_col, lag_cols, at_year=test_year)

    results = {}

    naive_origin_scores = []
    naive_origin_details = []
    for origin in origins:
        naive_val = get_eligible_split(prepared, target_cols, weight_col, lag_cols, at_year=origin)
        if naive_val.empty:
            continue
        naive_val_prediction = naive_predict(naive_val, naive_lag_cols)
        naive_val_metrics = score_composition(naive_val[target_cols], naive_val_prediction, target_cols)
        naive_origin_scores.append(naive_val_metrics['total_variation'])
        naive_origin_details.append(naive_val_metrics)

    if len(naive_origin_scores) != len(origins):
        raise RuntimeError(
            f'Naive baseline: expected {len(origins)} rolling origins, '
            f'but only {len(naive_origin_scores)} were successfully evaluated.'
        )

    results[('Naive baseline', 'validation')] = {
        'observations': sum(m['observations'] for m in naive_origin_details),
        'total_variation': float(np.mean(naive_origin_scores)),
        'n_origins': len(naive_origin_scores),
        'origin_scores_json': json.dumps(naive_origin_scores),
    }

    naive_test_prediction = naive_predict(test, naive_lag_cols)
    results[('Naive baseline', 'test')] = score_composition(test[target_cols], naive_test_prediction, target_cols)

    model_names = ['Ridge Regression', 'Random Forest', 'Gradient Boosting']

    rolling_selection = select_best_params_rolling(
        prepared, target_cols, weight_col, lag_cols, numeric_by_model,
        categorical, model_names, origins, is_composition=True,
    )

    for model_name in model_names:
        numeric_for_model = numeric_by_model[model_name]
        selection = rolling_selection[model_name]
        best_params = selection['best_params']

        results[(model_name, 'validation')] = {
            'observations': selection['pooled_validation_metrics']['observations'],
            'total_variation': selection['mean_validation_score'],
            'n_origins': len(selection['origin_scores']),
            'origin_scores_json': json.dumps(selection['origin_scores']),
        }

        final_model = build_model(model_name, best_params, numeric_for_model, categorical, multi_output=True)
        trainval_weights = trainval[weight_col].to_numpy()
        final_model.fit(
            trainval[numeric_for_model + categorical],
            shares_to_alr(trainval[target_cols]),
            model__sample_weight=trainval_weights
        )
        test_prediction = alr_to_shares(final_model.predict(test[numeric_for_model + categorical]))

        results[(model_name, 'test')] = score_composition(test[target_cols], test_prediction, target_cols)
        results[(model_name, 'best_params')] = best_params
        results[(model_name, 'fitted_model')] = final_model

    return results, prepared

### 2.1 Age strand -- overall activity level

In [51]:
age_overall = pd.read_csv(AGE_DATA_DIR / 'q3_age_overall_activity_level_panel.csv')
age_overall['LA_2023'] = age_overall['LA_2023'].astype('Int64').astype(str)

age_overall_targets = ['overall_inactive_rate', 'overall_fairly_active_rate', 'overall_active_rate']
age_overall_results, age_overall_panel = run_composition_task(
    age_overall,
    panel_keys=['LA_2023', 'age_group'],
    target_cols=age_overall_targets,
    weight_col='weighted_n_overall_activity_level',
    feature_level='overall',
)

for key, value in age_overall_results.items():
    if key[1] == 'test':
        clean = {k: (round(float(v), 4) if isinstance(v, (int, float, np.floating)) else v) for k, v in value.items()}
        print(key, clean)

('Naive baseline', 'test') {'observations': 256.0, 'total_variation': 0.1377, 'overall_mae': 0.0918, 'overall_rmse': 0.1432, 'overall_inactive_rate_mae': 0.1034, 'overall_inactive_rate_rmse': 0.162, 'overall_inactive_rate_r2': 0.0572, 'overall_fairly_active_rate_mae': 0.0616, 'overall_fairly_active_rate_rmse': 0.085, 'overall_fairly_active_rate_r2': -1.0996, 'overall_active_rate_mae': 0.1104, 'overall_active_rate_rmse': 0.1674, 'overall_active_rate_r2': 0.0992}
('Ridge Regression', 'test') {'observations': 256.0, 'total_variation': 0.1221, 'overall_mae': 0.0814, 'overall_rmse': 0.1366, 'overall_inactive_rate_mae': 0.0983, 'overall_inactive_rate_rmse': 0.1666, 'overall_inactive_rate_r2': 0.0026, 'overall_fairly_active_rate_mae': 0.0508, 'overall_fairly_active_rate_rmse': 0.0754, 'overall_fairly_active_rate_r2': -0.649, 'overall_active_rate_mae': 0.0952, 'overall_active_rate_rmse': 0.1501, 'overall_active_rate_r2': 0.2755}
('Random Forest', 'test') {'observations': 256.0, 'total_variatio

### 2.2 Disability strand -- overall activity level



In [52]:
dis_overall = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_overall_all_years.csv')
dis_overall['LA_2023'] = dis_overall['LA_2023'].astype('Int64').astype(str)

dis_overall_targets = ['inactive_rate', 'fairly_active_rate', 'active_rate']
dis_overall_results, dis_overall_panel = run_composition_task(
    dis_overall,
    panel_keys=['LA_2023', 'disability_group'],
    target_cols=dis_overall_targets,
    weight_col='weighted_n',
    feature_level='overall',
)

for key, value in dis_overall_results.items():
    if key[1] == 'test':
        clean = {k: (round(float(v), 4) if isinstance(v, (int, float, np.floating)) else v) for k, v in value.items()}
        print(key, clean)

('Naive baseline', 'test') {'observations': 504.0, 'total_variation': 0.2494, 'overall_mae': 0.1662, 'overall_rmse': 0.2454, 'inactive_rate_mae': 0.1934, 'inactive_rate_rmse': 0.2674, 'inactive_rate_r2': -0.7096, 'fairly_active_rate_mae': 0.0998, 'fairly_active_rate_rmse': 0.1608, 'fairly_active_rate_r2': -1.1113, 'active_rate_mae': 0.2055, 'active_rate_rmse': 0.2887, 'active_rate_r2': -0.853}
('Ridge Regression', 'test') {'observations': 504.0, 'total_variation': 0.2079, 'overall_mae': 0.1386, 'overall_rmse': 0.2029, 'inactive_rate_mae': 0.1564, 'inactive_rate_rmse': 0.2132, 'inactive_rate_r2': -0.0867, 'fairly_active_rate_mae': 0.0797, 'fairly_active_rate_rmse': 0.1359, 'fairly_active_rate_r2': -0.5084, 'active_rate_mae': 0.1797, 'active_rate_rmse': 0.2442, 'active_rate_r2': -0.3258}
('Random Forest', 'test') {'observations': 504.0, 'total_variation': 0.1989, 'overall_mae': 0.1326, 'overall_rmse': 0.1961, 'inactive_rate_mae': 0.1489, 'inactive_rate_rmse': 0.2039, 'inactive_rate_r2': 

### 2.3 Disability strand -- activity-specific MEMS7GR tiers



In [53]:
dis_level = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_MEMS7GR_all_years.csv')
dis_level['LA_2023'] = dis_level['LA_2023'].astype('Int64').astype(str)

dis_level_targets = ['inactive_rate', 'fairly_active_rate', 'active_rate']
dis_level_results, dis_level_panel = run_composition_task(
    dis_level,
    panel_keys=['LA_2023', 'disability_group', 'activity'],
    target_cols=dis_level_targets,
    weight_col='weighted_n',
    feature_level='activity',
)

for key, value in dis_level_results.items():
    if key[1] == 'test':
        clean = {k: (round(float(v), 4) if isinstance(v, (int, float, np.floating)) else v) for k, v in value.items()}
        print(key, clean)

('Naive baseline', 'test') {'observations': 62392.0, 'total_variation': 0.0114, 'overall_mae': 0.0076, 'overall_rmse': 0.0442, 'inactive_rate_mae': 0.0103, 'inactive_rate_rmse': 0.0529, 'inactive_rate_r2': 0.1265, 'fairly_active_rate_mae': 0.0056, 'fairly_active_rate_rmse': 0.0351, 'fairly_active_rate_r2': -0.6335, 'active_rate_mae': 0.0069, 'active_rate_rmse': 0.0427, 'active_rate_r2': -0.0694}
('Ridge Regression', 'test') {'observations': 62392.0, 'total_variation': 0.0091, 'overall_mae': 0.0061, 'overall_rmse': 0.038, 'inactive_rate_mae': 0.0089, 'inactive_rate_rmse': 0.0486, 'inactive_rate_r2': 0.2625, 'fairly_active_rate_mae': 0.004, 'fairly_active_rate_rmse': 0.0268, 'fairly_active_rate_r2': 0.0494, 'active_rate_mae': 0.0053, 'active_rate_rmse': 0.0355, 'active_rate_r2': 0.2595}
('Random Forest', 'test') {'observations': 62392.0, 'total_variation': 0.0083, 'overall_mae': 0.0055, 'overall_rmse': 0.0363, 'inactive_rate_mae': 0.0081, 'inactive_rate_rmse': 0.046, 'inactive_rate_r2': 

## 3. Task type 2: single participation rate (MONTHS_12 / DAYS10P60GR)



In [54]:
def run_single_rate_task(
    frame, panel_keys, target_col, weight_col,
    origins=ROLLING_ORIGINS, test_year=8, feature_level='activity'
):
    if feature_level == 'overall':
        prepared = add_lag_features(frame, panel_keys, [target_col], lags=(1, 2))
        extra_cols = [f'{target_col}_lag2']
    else:
        prepared = add_lag_features(frame, panel_keys, [target_col], lags=(1,), rolling_window=2)
        extra_cols = [f'{target_col}_roll2']

    group_col = panel_keys[1]
    prepared, interaction_cols = add_interaction_terms(prepared, group_col)

    lag_col = f'{target_col}_lag1'
    naive_lag_cols = [lag_col]
    lag_cols = naive_lag_cols + extra_cols
    categorical = list(panel_keys)
    numeric = lag_cols + ['time_trend', 'is_covid_year']

    numeric_by_model = {
        'Ridge Regression': numeric + interaction_cols,
        'Random Forest': numeric,
        'Gradient Boosting': numeric,
    }

    trainval = get_eligible_split(prepared, [target_col], weight_col, naive_lag_cols, before_year=test_year)
    test = get_eligible_split(prepared, [target_col], weight_col, lag_cols, at_year=test_year)

    results = {}

    naive_origin_scores = []
    naive_origin_details = []
    for origin in origins:
        naive_val = get_eligible_split(prepared, [target_col], weight_col, lag_cols, at_year=origin)
        if naive_val.empty:
            continue
        naive_val_prediction = naive_predict(naive_val, naive_lag_cols)
        naive_val_metrics = score_single_rate(naive_val[target_col], naive_val_prediction, target_col)
        naive_origin_scores.append(naive_val_metrics[f'{target_col}_mae'])
        naive_origin_details.append(naive_val_metrics)

    if len(naive_origin_scores) != len(origins):
        raise RuntimeError(
            f'Naive baseline: expected {len(origins)} rolling origins, '
            f'but only {len(naive_origin_scores)} were successfully evaluated.'
        )

    results[('Naive baseline', 'validation')] = {
        'observations': sum(m['observations'] for m in naive_origin_details),
        f'{target_col}_mae': float(np.mean(naive_origin_scores)),
        'n_origins': len(naive_origin_scores),
        'origin_scores_json': json.dumps(naive_origin_scores),
    }

    naive_test_prediction = naive_predict(test, naive_lag_cols)
    results[('Naive baseline', 'test')] = score_single_rate(test[target_col], naive_test_prediction, target_col)

    model_names = ['Ridge Regression', 'Random Forest', 'Gradient Boosting']

    rolling_selection = select_best_params_rolling(
        prepared, [target_col], weight_col, lag_cols, numeric_by_model,
        categorical, model_names, origins, is_composition=False,
    )

    for model_name in model_names:
        numeric_for_model = numeric_by_model[model_name]
        selection = rolling_selection[model_name]
        best_params = selection['best_params']

        results[(model_name, 'validation')] = {
            'observations': selection['pooled_validation_metrics']['observations'],
            f'{target_col}_mae': selection['mean_validation_score'],
            'n_origins': len(selection['origin_scores']),
            'origin_scores_json': json.dumps(selection['origin_scores']),
        }

        final_model = build_model(model_name, best_params, numeric_for_model, categorical, multi_output=False)
        trainval_weights = trainval[weight_col].to_numpy()
        final_model.fit(
            trainval[numeric_for_model + categorical],
            trainval[target_col],
            model__sample_weight=trainval_weights
        )
        test_prediction = np.clip(final_model.predict(test[numeric_for_model + categorical]), 0, 1)

        results[(model_name, 'test')] = score_single_rate(test[target_col], test_prediction, target_col)
        results[(model_name, 'best_params')] = best_params
        results[(model_name, 'fitted_model')] = final_model

    return results, prepared, test

### 3.1 Disability strand -- MONTHS_12 and DAYS10P60GR



In [55]:
dis_dm = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_days_months_all_years.csv')
dis_dm['LA_2023'] = dis_dm['LA_2023'].astype('Int64').astype(str)

months12_results, months12_panel, months12_test = run_single_rate_task(
    dis_dm, ['LA_2023', 'disability_group', 'activity'],
    target_col='participation_MONTHS_12', weight_col='weighted_n_MONTHS_12',
    feature_level='activity',
)
days_results, days_panel, days_test = run_single_rate_task(
    dis_dm, ['LA_2023', 'disability_group', 'activity'],
    target_col='participation_DAYS10P60GR', weight_col='weighted_n_DAYS10P60GR',
    feature_level='activity',
)

for label, results in [('MONTHS_12', months12_results), ('DAYS10P60GR', days_results)]:
    print(f'--- {label} ---')
    for key, value in results.items():
        if key[1] == 'test':
            clean = {k: (round(float(v), 4) if isinstance(v, (int, float, np.floating)) else v) for k, v in value.items()}
            print(key, clean)


--- MONTHS_12 ---
('Naive baseline', 'test') {'observations': 62392.0, 'participation_MONTHS_12_mae': 0.0224, 'participation_MONTHS_12_rmse': 0.0788, 'participation_MONTHS_12_r2': 0.3746}
('Ridge Regression', 'test') {'observations': 62392.0, 'participation_MONTHS_12_mae': 0.0212, 'participation_MONTHS_12_rmse': 0.0609, 'participation_MONTHS_12_r2': 0.626}
('Random Forest', 'test') {'observations': 62392.0, 'participation_MONTHS_12_mae': 0.0228, 'participation_MONTHS_12_rmse': 0.0586, 'participation_MONTHS_12_r2': 0.6541}
('Gradient Boosting', 'test') {'observations': 62392.0, 'participation_MONTHS_12_mae': 0.0212, 'participation_MONTHS_12_rmse': 0.0645, 'participation_MONTHS_12_r2': 0.5813}
--- DAYS10P60GR ---
('Naive baseline', 'test') {'observations': 62392.0, 'participation_DAYS10P60GR_mae': 0.0109, 'participation_DAYS10P60GR_rmse': 0.0547, 'participation_DAYS10P60GR_r2': 0.2277}
('Ridge Regression', 'test') {'observations': 62392.0, 'participation_DAYS10P60GR_mae': 0.0103, 'partic

### 3.2 Age strand -- months12_rate and days10p60gr_rate from the complete table



In [56]:
age_complete = pd.read_csv(AGE_DATA_DIR / 'q3_age_activity_participation_level_panel_complete.csv')
age_complete['LA_2023'] = age_complete['LA_2023'].astype('Int64').astype(str)

age_months12_results, age_months12_panel, age_months12_test = run_single_rate_task(
    age_complete, ['LA_2023', 'age_group', 'activity_suffix'],
    target_col='months12_rate', weight_col='weighted_n_months12',
    feature_level='activity',
)
age_days_results, age_days_panel, age_days_test = run_single_rate_task(
    age_complete, ['LA_2023', 'age_group', 'activity_suffix'],
    target_col='days10p60gr_rate', weight_col='weighted_n_days10p60gr',
    feature_level='activity',
)

age_complete_targets = ['activity_inactive_rate', 'activity_fairly_active_rate', 'activity_active_rate']
age_level_results, age_level_panel = run_composition_task(
    age_complete, panel_keys=['LA_2023', 'age_group', 'activity_suffix'],
    target_cols=age_complete_targets, weight_col='weighted_n_activity_level',
    feature_level='activity',
)

## 4. Extracting the highest predicted participation activity

In [57]:
def select_top_activity(
    working,
    id_cols,
    activity_col,
    prediction_col,
    top_n=1
):
    ranked = working.sort_values(
        id_cols + [prediction_col],
        ascending=(
            [True] * len(id_cols)
            + [False]
        )
    )

    return (
        ranked
        .groupby(
            id_cols,
            as_index=False
        )
        .head(top_n)[
            id_cols
            + [activity_col, prediction_col]
        ]
    )


validation_metric = (
    'participation_MONTHS_12_mae'
)

candidate_names = [
    'Naive baseline',
    'Ridge Regression',
    'Random Forest',
    'Gradient Boosting'
]

validation_mae = {
    model_name: months12_results[
        (model_name, 'validation')
    ][validation_metric]
    for model_name in candidate_names
}

best_disability_model_name = min(
    validation_mae,
    key=validation_mae.get
)

print(
    'Validation MAE by method:'
)

for model_name, mae_value in validation_mae.items():
    print(
        model_name,
        round(mae_value, 6)
    )

print(
    'Selected method for highest predicted participation activity:',
    best_disability_model_name
)

base_numeric_cols = [
    'participation_MONTHS_12_lag1',
    'participation_MONTHS_12_roll2',
    'time_trend',
    'is_covid_year'
]

categorical_cols = [
    'LA_2023',
    'disability_group',
    'activity'
]

if (
    best_disability_model_name
    == 'Naive baseline'
):
    preferred_working = (
        months12_test
        .dropna(
            subset=(
                categorical_cols
                + [
                    'participation_MONTHS_12_lag1'
                ]
            )
        )
        .copy()
    )

    preferred_working[
        'predicted_participation'
    ] = np.clip(
        preferred_working[
            'participation_MONTHS_12_lag1'
        ],
        0,
        1
    )

else:
    best_disability_model = months12_results[
        (
            best_disability_model_name,
            'fitted_model'
        )
    ]

    numeric_cols = list(base_numeric_cols)

    if (
        best_disability_model_name
        == 'Ridge Regression'
    ):
        interaction_cols = [
            column
            for column in months12_test.columns
            if column.startswith(
                'disability_group_x_time_'
            )
        ]

        numeric_cols = (
            numeric_cols
            + interaction_cols
        )

    preferred_working = (
        months12_test
        .dropna(
            subset=(
                numeric_cols
                + categorical_cols
            )
        )
        .copy()
    )

    preferred_working[
        'predicted_participation'
    ] = np.clip(
        best_disability_model.predict(
            preferred_working[
                numeric_cols
                + categorical_cols
            ]
        ),
        0,
        1
    )

top_activity_by_disability_group = (
    select_top_activity(
        preferred_working,
        id_cols=[
            'LA_2023',
            'disability_group'
        ],
        activity_col='activity',
        prediction_col=(
            'predicted_participation'
        ),
        top_n=1
    )
)

top_activity_by_disability_group.head(10)

Validation MAE by method:
Naive baseline 0.021806
Ridge Regression 0.020763
Random Forest 0.023831
Gradient Boosting 0.020939
Selected method for highest predicted participation activity: Ridge Regression


,LA_2023,disability_group,activity,predicted_participation
15,107,disty1,ACTTRAV_C03,0.653071
1007,107,disty10,ACTTRAV_C03,0.625171
1999,107,disty11,ACTTRAV_C03,0.613667
2991,107,disty12,ACTTRAV_C03,0.630091
3983,107,disty13,ACTTRAV_C03,0.618585
4975,107,disty2,ACTTRAV_C03,0.649432
5967,107,disty3,ACTTRAV_C03,0.641371
6959,107,disty4,ACTTRAV_C03,0.660114
7951,107,disty5,ACTTRAV_C03,0.681246
8943,107,disty6,ACTTRAV_C03,0.601932


## 5. Summary table



In [58]:
def collect_summary(results_dict, task_name):
    rows = []

    for (model_name, split), value in results_dict.items():
        if (
            split in ['validation', 'test']
            and isinstance(value, dict)
        ):
            rows.append({
                'task': task_name,
                'model': model_name,
                'split': split,
                **value
            })

    return pd.DataFrame(rows)


summary = pd.concat(
    [
        collect_summary(
            age_overall_results,
            'age_overall_level'
        ),
        collect_summary(
            dis_overall_results,
            'disability_overall_level'
        ),
        collect_summary(
            dis_level_results,
            'disability_activity_level'
        ),
        collect_summary(
            months12_results,
            'disability_months12'
        ),
        collect_summary(
            days_results,
            'disability_days10p60gr'
        ),
        collect_summary(
            age_months12_results,
            'age_months12'
        ),
        collect_summary(
            age_days_results,
            'age_days10p60gr'
        ),
        collect_summary(
            age_level_results,
            'age_activity_level'
        ),
    ],
    ignore_index=True
)

summary = summary.sort_values(
    ['task', 'split', 'model']
).reset_index(drop=True)

print(
    summary[
        ['task', 'model', 'split', 'observations']
    ]
)

summary

                        task              model       split  observations
0         age_activity_level  Gradient Boosting        test         31585
1         age_activity_level     Naive baseline        test         31585
2         age_activity_level      Random Forest        test         31585
3         age_activity_level   Ridge Regression        test         31585
4         age_activity_level  Gradient Boosting  validation        126870
..                       ...                ...         ...           ...
59  disability_overall_level   Ridge Regression        test           504
60  disability_overall_level  Gradient Boosting  validation          1976
61  disability_overall_level     Naive baseline  validation          1976
62  disability_overall_level      Random Forest  validation          1976
63  disability_overall_level   Ridge Regression  validation          1976

[64 rows x 4 columns]


,task,model,split,observations,total_variation,n_origins,origin_scores_json,overall_mae,overall_rmse,overall_inactive_rate_mae,...,days10p60gr_rate_r2,activity_inactive_rate_mae,activity_inactive_rate_rmse,activity_inactive_rate_r2,activity_fairly_active_rate_mae,activity_fairly_active_rate_rmse,activity_fairly_active_rate_r2,activity_active_rate_mae,activity_active_rate_rmse,activity_active_rate_r2
0,age_activity_level,Gradient Boosting,test,31585,0.008343,NaN,NaN,0.005562,0.024559,NaN,...,NaN,0.007781,0.030364,0.731242,0.003791,0.016729,0.499745,0.005114,0.024649,0.635443
1,age_activity_level,Naive baseline,test,31585,0.009492,NaN,NaN,0.006328,0.025180,NaN,...,NaN,0.008348,0.030253,0.733191,0.004883,0.020013,0.284075,0.005754,0.024212,0.648236
2,age_activity_level,Random Forest,test,31585,0.008034,NaN,NaN,0.005356,0.024105,NaN,...,NaN,0.007687,0.030727,0.724772,0.003801,0.016970,0.485264,0.004580,0.022607,0.693330
3,age_activity_level,Ridge Regression,test,31585,0.008938,NaN,NaN,0.005958,0.026514,NaN,...,NaN,0.008652,0.034334,0.656362,0.004214,0.018379,0.396247,0.005009,0.024340,0.644511
4,age_activity_level,Gradient Boosting,validation,126870,0.007462,4.0,"[0.008409938238655728, 0.006874335644802056, 0...",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,disability_overall_level,Ridge Regression,test,504,0.207858,NaN,NaN,0.138572,0.202944,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,disability_overall_level,Gradient Boosting,validation,1976,0.200502,4.0,"[0.19703922092291212, 0.2164232256735321, 0.20...",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,disability_overall_level,Naive baseline,validation,1976,0.266964,4.0,"[0.2659882223456809, 0.28156775070616297, 0.26...",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,disability_overall_level,Random Forest,validation,1976,0.207675,4.0,"[0.20912364243507545, 0.22297611700100933, 0.2...",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
import os
import pickle

output_dir = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs"
os.makedirs(output_dir, exist_ok=True)

# 1. summary table
summary.to_csv(os.path.join(output_dir, 'summary_full.csv'), index=False)

# 2. highest predicted participation activity table
top_activity_by_disability_group.to_csv(
    os.path.join(output_dir, 'top_activity_by_disability_group.csv'), index=False
)

# 3. every fitted model from every task
all_results = {
    'age_overall': age_overall_results,
    'dis_overall': dis_overall_results,
    'dis_level': dis_level_results,
    'months12': months12_results,
    'days': days_results,
    'age_months12': age_months12_results,
    'age_days': age_days_results,
    'age_level': age_level_results,
}

fitted_models = {}
for task_name, results in all_results.items():
    for (model_name, key), value in results.items():
        if key == 'fitted_model':
            fitted_models[f'{task_name}__{model_name}'] = value

with open(os.path.join(output_dir, 'fitted_models.pkl'), 'wb') as f:
    pickle.dump(fitted_models, f)

# 4. best_params for every task/model
best_params_rows = []
for task_name, results in all_results.items():
    for (model_name, key), value in results.items():
        if key == 'best_params':
            best_params_rows.append({'task': task_name, 'model': model_name, **value})

import pandas as pd
pd.DataFrame(best_params_rows).to_csv(
    os.path.join(output_dir, 'best_hyperparameters.csv'), index=False
)

print('Saved to', output_dir)
print('- summary_full.csv')
print('- top_activity_by_disability_group.csv')
print('- fitted_models.pkl', f'({len(fitted_models)} models)')
print('- best_hyperparameters.csv')

Saved to C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs
- summary_full.csv
- top_activity_by_disability_group.csv
- fitted_models.pkl (24 models)
- best_hyperparameters.csv


## 6. Activity coverage check

In [60]:
print("MEMS7GR activities:", dis_level["activity"].nunique())
print("DAYS10P60GR and MONTHS_12 activities:", dis_dm["activity"].nunique())

assert dis_level["activity"].nunique() == 124
assert dis_dm["activity"].nunique() == 124
assert "HULAHOOP_P27" not in set(dis_level["activity"])
assert "HULAHOOP_P27" not in set(dis_dm["activity"])

MEMS7GR activities: 124
DAYS10P60GR and MONTHS_12 activities: 124
